# Python 3.14.0 New Features Demonstration

This notebook demonstrates some of the key new features introduced in Python 3.14.0 (released October 7, 2025).

## 1. Template String Literals (t-strings)

Python 3.14 introduces template string literals (t-strings), a new string formatting syntax that provides a safer, more explicit alternative to f-strings. T-strings are denoted by a t prefix and use ${...} for interpolation instead of curly braces. Unlike f-strings, which produce a str object, t-strings resolve to a Template instance.

- **Explicit and Safe:** T-strings require explicit conversion and don't automatically call __str__() or __format__(), reducing security risks and unexpected behavior.
- **Template Syntax:** Uses ${expression} for interpolation, making it visually distinct from f-strings and aligned with common templating languages.
- **Deferred Evaluation:** Unlike f-strings which evaluate immediately, t-strings can be stored and evaluated later, making them useful for templates.

### Basic Syntax

In [3]:
# Traditional f-string
name = "Alice"
age = 30
print(f"Hello, {name}! You are {age} years old.")

# New t-string (Python 3.14+)
print(t"Hello, ${name}! You are ${age} years old.")

Hello, Alice! You are 30 years old.
Template(strings=('Hello, $', '! You are $', ' years old.'), interpolations=(Interpolation('Alice', 'name', None, ''), Interpolation(30, 'age', None, '')))


You process t-strings by iterating over their components, using attributes such as .strings, .interpolations, and .values for safe and customized handling.

### Simple Interpolation

In [9]:
# Basic variable interpolation
username = "bob"
message = t"Welcome, ${username}!"
print(message) 

# Expressions inside t-strings
x = 10
y = 20
result = t"The sum of ${x} and ${y} is ${x + y}"
print(result)  

Template(strings=('Welcome, $', '!'), interpolations=(Interpolation('bob', 'username', None, ''),))
Template(strings=('The sum of $', ' and $', ' is $', ''), interpolations=(Interpolation(10, 'x', None, ''), Interpolation(20, 'y', None, ''), Interpolation(30, 'x + y', None, '')))


### Safety and Explicit Conversion

In [8]:
class User:
    def __init__(self, name):
        self.name = name
    
    def __str__(self):
        return f"User({self.name})"

user = User("Charlie")

# F-string automatically calls __str__
print(f"User: {user}")  

# T-string requires explicit conversion
# print(t"User: ${user}")  # This would raise an error!
print(t"User: ${str(user)}")  

User: User(Charlie)
Template(strings=('User: $', ''), interpolations=(Interpolation('User(Charlie)', 'str(user)', None, ''),))


### SQL Query Templates (Safer)

In [10]:
# T-strings for SQL templates (illustration)
table = "users"
column = "email"

# More explicit about what's being interpolated
query = t"SELECT ${column} FROM ${table} WHERE active = true"
print(query)

Template(strings=('SELECT $', ' FROM $', ' WHERE active = true'), interpolations=(Interpolation('email', 'column', None, ''), Interpolation('users', 'table', None, '')))


## 2. Deferred Evaluation of Annotations

Python 3.14 makes deferred evaluation of annotations the default behavior. This means type annotations are no longer evaluated at function/class definition time but are stored as strings and evaluated only when needed.

**What Changed**
- **Before Python 3.14:** Annotations were evaluated immediately when the code was parsed, which could cause NameError if types weren't defined yet.
- **Python 3.14+:** Annotations are automatically stored as strings and evaluated lazily, similar to using from __future__ import annotations in earlier versions.

### Forward References (Self-Referencing Classes)

In [ ]:
class TreeNode:
    def __init__(self, value: int):
        self.value = value
        self.left: TreeNode | None = None   # Can reference TreeNode before fully defined
        self.right: TreeNode | None = None
    
    def add_child(self, node: TreeNode) -> TreeNode:
        # TreeNode is referenced before the class definition completes
        return node

# Before Python 3.14, you would need:
# - Quotes: left: "TreeNode"
# - Or: from __future__ import annotations

### Circular Type Dependencies

In [13]:
class User:
    def __init__(self, name: str):
        self.name = name
        self.posts: list[Post] = []  # Post not defined yet - no problem!
    
    def add_post(self, post: Post) -> None:
        self.posts.append(post)

class Post:
    def __init__(self, title: str, author: User):
        self.title = title
        self.author: User = author  # Mutual reference works!
    
    def get_author(self) -> User:
        return self.author

# Before Python 3.14: This would cause NameError
# You'd need to use strings: posts: list["Post"]

In [14]:
# Create a user
user1 = User("Alice")

# Create a post authored by that user
post1 = Post("My first post", user1)

# Link them together
user1.add_post(post1)

# Access relationships
print(post1.get_author().name)  # Output: Alice
print(user1.posts[0].title)     # Output: My first post

Alice
My first post


### Accessing Annotations

In [15]:
import typing

class Person:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age
    
    def greet(self, other: Person) -> str:
        return f"Hello, {other.name}!"

# Annotations are stored as strings
print(Person.greet.__annotations__)

# To evaluate them, use typing.get_type_hints()
hints = typing.get_type_hints(Person.greet)
print(hints)

{'other': <class '__main__.Person'>, 'return': <class 'str'>}
{'other': <class '__main__.Person'>, 'return': <class 'str'>}


### Practical Example: API Response Models

In [17]:
from datetime import datetime
from typing import Optional

# Python 3.14 - Clean, readable code without workarounds
class Comment:
    def __init__(self, text: str, author: User):
        self.text = text
        self.author = author
        self.timestamp: datetime = datetime.now()

class BlogPost:
    def __init__(self, title: str, author: User):
        self.title = title
        self.author = author
        self.comments: list[Comment] = []
    
    def add_comment(self, comment: Comment) -> BlogPost:
        self.comments.append(comment)
        return self

class User:
    def __init__(self, username: str):
        self.username = username
        self.posts: list[BlogPost] = []
        self.followers: list[User] = []  # Self-reference!
    
    def follow(self, other: User) -> None:
        self.followers.append(other)

# All forward references and circular dependencies work seamlessly!

**Key Features:**

- 👉 author: User works even though User is defined later in the file — because Python 3.14 supports “clean forward references” natively.
- add_comment() appends a comment and returns the same BlogPost object, allowing method chaining
- followers: List of other User objects following this one — this is a self-referential type hint.
- Notice the mutual references:
    - Comment → User
    - BlogPost → User, Comment
    - User → BlogPost, User

In Python 3.14, these circular or forward references just work, thanks to the new type hint evaluation mechanism that defers resolution automatically — no need for quotes or from __future__ import annotations.

## 3. Subinterpreters (PEP 734)

Python 3.14 adds support for running code in separate subinterpreters, enabling true parallelism.

Subinterpreters are isolated Python execution environments within a single process. Each subinterpreter has its own:

-   Global namespace
-   Import system
-   Module state
-   GIL (as of Python 3.14)

This isolation allows multiple Python interpreters to run simultaneously without interfering with each other.

### Creating and Running Code in Subinterpreters

In [38]:
import _interpreters as interpreters
import textwrap

# Create a new subinterpreter
interp = interpreters.create()

# Run code in the subinterpreter
code = textwrap.dedent("""
    print("Hello from subinterpreter!")
    x = 10 + 20
    print(f"Calculation result: {x}")
""")

interpreters.run_string(interp, code)

Hello from subinterpreter!
Calculation result: 30


### Parallel CPU-Bound Tasks

In [39]:
# Use existing _interpreters and textwrap imports from other cells
# (avoid re-importing the same module in multiple cells)

def cpu_intensive_task(n):
    """Simulate CPU-bound work"""
    code = textwrap.dedent(f"""
    import time
    start = time.time()
    result = sum(i*i for i in range({n}))
    end = time.time()
    print(f"\\nSubinterp calculated {{result}} in {{end-start:.2f}}s")
    """)
    return code

# Create multiple subinterpreters
interp1 = interpreters.create()
interp2 = interpreters.create()

# Run CPU-intensive tasks in parallel
start = time.time()
interpreters.run_string(interp1, cpu_intensive_task(10_000_000))
interpreters.run_string(interp2, cpu_intensive_task(10_000_000))
end = time.time()

print(f"\nTotal time with subinterpreters: {end-start:.2f}s")


Subinterp calculated 333333283333335000000 in 0.53s

Total time with subinterpreters: 1.06s
Subinterp calculated 333333283333335000000 in 0.53s



### Practical Example: Web Scraping in Parallel

In [42]:
import _interpreters as interpreters
import _interpqueues as queues

# Create result queue (maxsize required; 0 means unbounded)
result_queue = queues.create(maxsize=0)

urls = [
    "https://docs.python.org/3/library/index.html",
    "https://docs.python.org/3/extending/index.html",
    "https://docs.python.org/3/c-api/index.html"
]

# Create subinterpreter for each URL
for i, url in enumerate(urls):
    interp = interpreters.create()
    code = f"""
import _interpqueues as queues
import urllib.request

url = "{url}"
try:
    with urllib.request.urlopen(url) as response:
        content_length = len(response.read())
        result = f"{{url}}: {{content_length}} bytes"
except Exception as e:
    result = f"{{url}}: Error - {{e}}"

queues.put({result_queue}, result.encode())
"""
    interpreters.run_string(interp, code)

# Collect results
for _ in range(len(urls)):
    payload, _ = queues.get(result_queue)
    print(payload.decode())

https://docs.python.org/3/library/index.html: 77115 bytes
https://docs.python.org/3/extending/index.html: 23917 bytes
https://docs.python.org/3/c-api/index.html: 26687 bytes


## 4. Improved Error Messages

Python 3.14 introduces even more helpful error messages that continue the trend of making Python more beginner-friendly and easier to debug. These improvements build upon the enhanced error messages introduced in Python 3.10 and 3.11.

### Better AttributeError Messages for Method Calls

Python 3.14 now suggests the correct method name when you accidentally call a method that doesn't exist, including distinguishing between instance and class methods.

In [18]:
# Incorrect method name
text = "hello world"
result = text.uppercases()

# Python 3.13 and earlier:**
# AttributeError: 'str' object has no attribute 'uppercases'

#Python 3.14:**

AttributeError: 'str' object has no attribute 'uppercases'

### Improved NameError Suggestions

When you misspell a variable name, Python 3.14 provides smarter suggestions, even for variables in different scopes.

In [19]:
def calculate_total():
    item_price = 100
    quantity = 5
    # Typo in variable name
    total = itemprice * quantity
    return total

calculate_total()

NameError: name 'itemprice' is not defined

### Better TypeError Messages for Function Calls

More descriptive messages when passing wrong argument types or counts.

In [20]:
def greet(name, age):
    return f"Hello {name}, you are {age} years old"

# Missing argument
greet("Alice")

TypeError: greet() missing 1 required positional argument: 'age'

### Enhanced Import Error Messages

Clearer messages when imports fail, with suggestions for common mistakes.

In [21]:
from collections import DefaultDict  # Wrong capitalization

ImportError: cannot import name 'DefaultDict' from 'collections' (/opt/homebrew/Cellar/python@3.14/3.14.0_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/collections/__init__.py)

## 5. Zstandard Compression Support (PEP 784)

Python 3.14 adds native support for Zstandard (zstd) compression, a modern compression algorithm developed by Facebook that offers excellent compression ratios and very fast compression/decompression speeds.

**What is Zstandard?**
Zstandard is a real-time compression algorithm that provides:

-   Better compression ratios than gzip/zlib at comparable speeds
-   Extremely fast decompression
-   Adjustable compression levels (1-22, with higher = better compression but slower)
-   Dictionary compression support for small data

### Simple Compression and Decompression

In [2]:
# Install missing dependency in this notebook environment
%pip install zstandard

Note: you may need to restart the kernel to use updated packages.


In [3]:
import zstandard as zstd

# Compress data
original_data = b"Hello, World! " * 100
compressor = zstd.ZstdCompressor()
compressed = compressor.compress(original_data)

print(f"Original size: {len(original_data)} bytes")
print(f"Compressed size: {len(compressed)} bytes")
print(f"Compression ratio: {len(original_data)/len(compressed):.2f}x")

# Decompress data
decompressor = zstd.ZstdDecompressor()
decompressed = decompressor.decompress(compressed)
assert decompressed == original_data

Original size: 1400 bytes
Compressed size: 31 bytes
Compression ratio: 45.16x


### Compression Levels

In [4]:
data = b"Python 3.14 adds zstandard support! " * 50

# Try different compression levels
for level in [1, 3, 10, 19]:
    compressor = zstd.ZstdCompressor(level=level)
    compressed = compressor.compress(data)
    print(f"Level {level:2d}: {len(compressed)} bytes")

Level  1: 55 bytes
Level  3: 55 bytes
Level 10: 55 bytes
Level 19: 55 bytes


### Streaming Compression (for large files)

In [7]:
import zstandard as zstd

# Compress a file
with open('input.txt', 'rb') as input_file:
    with open('output.txt.zst', 'wb') as output_file:
        compressor = zstd.ZstdCompressor()
        compressor.copy_stream(input_file, output_file)

# Decompress a file
with open('output.txt.zst', 'rb') as input_file:
    with open('decompressed.txt', 'wb') as output_file:
        decompressor = zstd.ZstdDecompressor()
        decompressor.copy_stream(input_file, output_file)

### Comparison with Other Formats

In [8]:
import zstandard as zstd
import gzip
import bz2

data = b"Python compression comparison " * 1000

# Zstandard
zstd_compressed = zstd.ZstdCompressor().compress(data)

# Gzip
gzip_compressed = gzip.compress(data)

# Bzip2
bz2_compressed = bz2.compress(data)

print(f"Original:  {len(data)} bytes")
print(f"Zstandard: {len(zstd_compressed)} bytes ({len(data)/len(zstd_compressed):.2f}x)")
print(f"Gzip:      {len(gzip_compressed)} bytes ({len(data)/len(gzip_compressed):.2f}x)")
print(f"Bzip2:     {len(bz2_compressed)} bytes ({len(data)/len(bz2_compressed):.2f}x)")

Original:  30000 bytes
Zstandard: 43 bytes (697.67x)
Gzip:      137 bytes (218.98x)
Bzip2:     105 bytes (285.71x)


Speed: Zstandard is typically 3-5x faster than gzip at similar compression ratios

## 6. Free-Threaded Python (No-GIL)

Python 3.14 officially supports free-threaded builds without the Global Interpreter Lock.

In [ ]:
import sys
import threading
import time

# Check if running in free-threaded mode
is_free_threaded = hasattr(sys, '_is_gil_enabled') and not sys._is_gil_enabled()

print(f"Free-threaded mode: {is_free_threaded}")
print(f"Python version: {sys.version}")

def cpu_bound_work(n):
    """CPU-intensive task"""
    result = 0
    for i in range(n):
        result += i ** 2
    return result

# Demonstrate threading (benefits more in free-threaded mode)
start_time = time.time()

threads = []
for i in range(4):
    t = threading.Thread(target=cpu_bound_work, args=(1000000,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

elapsed = time.time() - start_time

print(f"\nCompleted 4 CPU-bound tasks in {elapsed:.3f} seconds")
if is_free_threaded:
    print("Running in free-threaded mode - true parallelism achieved!")
else:
    print("Running with GIL - threads execute sequentially for CPU-bound tasks")

## 7. Incremental Garbage Collection

Python 3.14 introduces incremental garbage collection to reduce pause times.

In [ ]:
import gc
import time

# Check garbage collection stats
print("Garbage Collection Information:")
print(f"GC is enabled: {gc.isenabled()}")
print(f"GC thresholds: {gc.get_threshold()}")
print(f"GC counts: {gc.get_count()}")

# Create some objects to demonstrate GC
large_list = []
for i in range(10000):
    large_list.append({'id': i, 'data': [j for j in range(100)]})

print(f"\nCreated {len(large_list)} objects")

# Force garbage collection and time it
start = time.perf_counter()
collected = gc.collect()
duration = time.perf_counter() - start

print(f"Objects collected: {collected}")
print(f"GC duration: {duration*1000:.3f} ms")
print("\nIn Python 3.14, incremental GC reduces pause times for large heaps!")

## 8. REPL Improvements

Python 3.14 includes syntax highlighting and improved autocompletion in the interactive shell.

In [ ]:
print("New REPL Features in Python 3.14:")
print("\n1. Syntax Highlighting:")
print("   - Keywords, strings, and numbers are colorized in real-time")
print("   - Customizable color themes")
print("\n2. Import Autocompletion:")
print("   - Tab completion now works for module names")
print("   - Example: 'from coll<TAB>' suggests 'collections'")
print("\n3. Better History:")
print("   - Improved navigation through command history")
print("   - Persistent history across sessions")
print("\n4. Colorized Output:")
print("   - unittest now has colored output like pytest")
print("   - pdb includes syntax highlighting")

## Summary

Python 3.14.0 brings significant improvements:

### Performance
- **JIT Compiler**: 3-5% performance boost on x86-64 and AArch64
- **Free-threaded Python**: No-GIL builds for true parallelism
- **Incremental GC**: Reduced pause times for large applications

### Developer Experience
- **T-strings**: Custom string processing with familiar syntax
- **Better error messages**: Helpful suggestions for typos
- **REPL improvements**: Syntax highlighting and better completion

### New Features
- **Deferred annotations**: Better performance and forward references
- **Subinterpreters**: True parallelism within Python
- **Zstandard compression**: Modern compression algorithm support
- **Safe debugger interface**: Attach debuggers without stopping processes

These features make Python 3.14 faster, more developer-friendly, and better suited for modern multi-core systems!